Qima HR Payroll ingestion pipeline — SharePoint to Snowflake RAW layer


In [ ]:
%%sql -r dataframe_1
USE WAREHOUSE SANDBOX_WH;
USE DATABASE SANDBOX_DB;
USE SCHEMA HR_PAYROLL_QIMA;

In [ ]:
from datetime import datetime, timedelta, timezone
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Compute the ingestion time window [FROM_TS, TO_TS)

OFFSET_MINUTES = 5  # safety buffer — skip files SharePoint may still be writing

row = session.sql("""
    SELECT COALESCE(
        MAX(SHAREPOINT_MODIFIED_AT),
        '2024-01-01'::TIMESTAMP_TZ
    ) AS HWM
    FROM FILE_LOAD
    WHERE INGEST_STATUS = 'SUCCESS'
""").collect()

FROM_TS = row[0]['HWM']
TO_TS   = session.sql(f"""
    SELECT TIMESTAMPADD('MINUTE', -{OFFSET_MINUTES}, CURRENT_TIMESTAMP())
""").collect()[0][0]

# Guard: if window is zero or negative, stop early — nothing to do
if FROM_TS >= TO_TS:
    raise RuntimeError(f'Empty window: FROM_TS={FROM_TS} >= TO_TS={TO_TS}. Nothing to ingest.')

In [ ]:
%%sql -r dataframe_2
CREATE TEMPORARY STAGE IF NOT EXISTS TEMP_PAYROLL_STAGE
    COMMENT = 'Session-scoped stage for raw payroll file bytes. Auto-dropped at session end.';

In [ ]:
import json

# Call the stored procedure — auth, pagination, and filtering all happen inside it
# Skip if candidates already loaded (SP walks SharePoint folders and can be slow)
try:
    candidates
    print('(reusing cached candidates from previous run)')
except NameError:
    result = session.sql(f"""
        CALL SP_LIST_SHAREPOINT_FILES('{FROM_TS}'::TIMESTAMP_TZ, '{TO_TS}'::TIMESTAMP_TZ)
    """).collect()
    candidates = json.loads(result[0][0])

print(f'{len(candidates)} candidate(s) in window [{FROM_TS}, {TO_TS})\n')
for i, c in enumerate(candidates, 1):
    print(f"  [{i}] {c['name']}")
    print(f"      id          : {c['id']}")
    print(f"      path        : {c.get('path', 'N/A')}")
    print(f"      size_bytes  : {c['size_bytes']:,}")
    print(f"      modified_at : {c['modified_at']}")
    print()

if not candidates:
    print('No new/modified files to ingest — pipeline will finish cleanly.')

In [ ]:
import json

result = session.sql(f"""
    CALL SP_DOWNLOAD_AND_STAGE_FILES(
        $${json.dumps(candidates)}$$,
        'TEMP_PAYROLL_STAGE'
    )
""").collect()

stage_result = json.loads(result[0][0])

print(f"Staged {stage_result['staged']}/{stage_result['total']} file(s)\n")
for f in stage_result['files']:
    icon = 'OK' if f['result'] == 'OK' else 'FAIL'
    print(f"  [{icon}] {f['name']}")
    if f['result'] == 'OK':
        print(f"        downloaded : {f['downloaded_bytes']:,} bytes")
    else:
        print(f"        error     : {f['result']}")
    print()

# Verify staged files
print('--- Staged files ---')
staged_files = session.sql('LIST @TEMP_PAYROLL_STAGE').collect()
for sf in staged_files:
    print(f"  {sf['name']}  ({sf['size']:,} bytes)")